Notebook to generate the ranking based on the ratio between the model and baseline WIS

In [1]:
import numpy as np 
import pandas as pd 
from aux_func import code_to_state, estado_para_regiao

In [2]:
challenge = 'chik_state'

df_preds = pd.read_csv(f'./predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,2.743512,3.561896,5.283421,9.016248,17.670073,40.444549,89.679029,143.716210,206.231430,33,5354,1,82.77,3rd_imdc_emap_lstm
1,2022-10-16,3.743952,5.428969,6.920945,12.201003,27.237822,55.050853,112.596736,216.009888,325.011995,33,5354,1,82.77,3rd_imdc_emap_lstm
2,2022-10-23,4.875423,5.693745,8.176445,12.618417,22.377075,42.060248,77.746351,114.612724,173.066908,33,5354,1,82.77,3rd_imdc_emap_lstm
3,2022-10-30,3.250737,4.192324,5.602107,9.793079,17.521617,34.101676,68.882955,117.519627,166.359700,33,5354,1,82.77,3rd_imdc_emap_lstm
4,2022-11-06,2.961620,3.732665,5.172567,9.438648,20.027886,46.905846,103.815092,215.222367,336.458355,33,5354,1,82.77,3rd_imdc_emap_lstm


In [3]:
df_preds.model.unique()

<ArrowStringArray>
[                      '3rd_imdc_emap_lstm',
 '3rd_imdc_emap_epidematicos_prophet_fixed',
 '3rd_imdc_emap_epidematicos_sarimax_fixed',
          '3rd_imdc_lncc_ARp26_chikungunya',
                  '3rd_imdc_emap_xgbsillas',
                          'DS-OKSTATE-2026',
              '3rd_imdc_purdue_neuralearth',
                    '3rd_imdc_nus_nus-cerm',
  '3rd_imdc_lncc_surge_model26_chikungunya',
                  '3rd_imdc_procc_bb_model',
      '3rd_imdc_lncc_clidengo26chikungunya']
Length: 11, dtype: str

In [4]:
df_preds.model.unique().shape

(11,)

In [5]:
df_agg_wis = (
    df_preds
    .groupby(["model", "adm_1", "validation"], as_index=False)["wis"]
    .mean()
)


df_agg_wis.head()

,model,adm_1,validation,wis
0,3rd_imdc_emap_epidematicos_prophet_fixed,11,1,1.61
1,3rd_imdc_emap_epidematicos_prophet_fixed,11,2,2.55
2,3rd_imdc_emap_epidematicos_prophet_fixed,11,3,71.58
3,3rd_imdc_emap_epidematicos_prophet_fixed,11,4,11.59
4,3rd_imdc_emap_epidematicos_prophet_fixed,12,1,0.82


Gerando um ranking da média da diferença entre os modelos e o baseline: 

In [6]:
model_baseline = '3rd_imdc_procc_bb_model'

df_baseline = (
    df_agg_wis.loc[df_agg_wis["model"] == model_baseline,
           ["adm_1", "validation", "wis"]]
    .rename(columns={"wis": "wis_baseline"})
)

# Junta o WIS do baseline aos demais modelos
df_ratio = df_agg_wis.merge(
    df_baseline,
    on=["adm_1", "validation"],
    how="left"
)

# Razão WIS / Baseline
df_ratio["wis_ratio"] = (
    df_ratio["wis"] / df_ratio["wis_baseline"]
)

# Médias das razões
df_summary = (
    df_ratio
    .groupby(["adm_1", "model"], as_index=False)
    .agg(
        arithmetic_mean_ratio=("wis_ratio", "mean"),
        geometric_mean_ratio=("wis_ratio", lambda x: np.exp(np.mean(np.log(x)))),
    )
    .sort_values(["adm_1", "geometric_mean_ratio"])
)

df_summary.head()

,adm_1,model,arithmetic_mean_ratio,geometric_mean_ratio
8,11,3rd_imdc_procc_bb_model,1.000000,1.000000
7,11,3rd_imdc_nus_nus-cerm,1.048951,1.036956
9,11,3rd_imdc_purdue_neuralearth,1.101244,1.068451
2,11,3rd_imdc_emap_lstm,1.083302,1.079902
10,11,DS-OKSTATE-2026,1.231481,1.149659


In [7]:
df_ratio['region'] = df_ratio['adm_1'].replace(code_to_state).replace(estado_para_regiao)

df_ratio.to_csv(f'predictions/rank_ratio_{challenge}.csv', index = False)

In [11]:
df_ratio.model.unique()

<ArrowStringArray>
['3rd_imdc_emap_epidematicos_prophet_fixed',
 '3rd_imdc_emap_epidematicos_sarimax_fixed',
                       '3rd_imdc_emap_lstm',
                  '3rd_imdc_emap_xgbsillas',
          '3rd_imdc_lncc_ARp26_chikungunya',
      '3rd_imdc_lncc_clidengo26chikungunya',
  '3rd_imdc_lncc_surge_model26_chikungunya',
                    '3rd_imdc_nus_nus-cerm',
                  '3rd_imdc_procc_bb_model',
              '3rd_imdc_purdue_neuralearth',
                          'DS-OKSTATE-2026']
Length: 11, dtype: str